# EVREN CPU ortam kontrolü

Çalışma Alanı erişimi açıldığında CPU runtime içinde çalıştırın.
Bu notebook yalnız sürüm ve kaynak sınırı metadatasını okur; GPU işi başlatmaz.
Çıktı bir eğitim izni veya maliyet ölçümü değildir.


In [ ]:
import importlib.metadata
import json
import platform
from pathlib import Path, PurePosixPath

result = {"schema": "evren-cpu-readiness-v1", "python": platform.python_version(),
          "gpu_calls": 0, "models_created": 0, "packages": {}}
for package in ("numpy", "Pillow", "torch", "onnx", "onnxruntime"):
    try:
        result["packages"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        result["packages"][package] = "unavailable"

try:
    with Path("/proc/self/cgroup").open("rb") as stream:
        raw = stream.read(4097)
    if len(raw) > 4096:
        raise ValueError("cgroup bound")
    identities = [line[3:] for line in raw.decode().splitlines() if line.startswith("0::")]
    if len(identities) != 1:
        raise ValueError("cgroup identity")
    relative = PurePosixPath(identities[0])
    if not relative.is_absolute() or ".." in relative.parts or len(relative.parts) > 32:
        raise ValueError("cgroup identity")
    folder = Path("/sys/fs/cgroup").joinpath(*relative.parts[1:])
    counters = {}
    for key, name in (("ram_limit_bytes", "memory.max"),
                      ("swap_limit_bytes", "memory.swap.max")):
        path = folder / name
        if any(p.is_symlink() for p in (path, *path.parents)):
            raise ValueError("cgroup ancestry")
        with path.open("rb") as stream:
            value = stream.read(33)
        if len(value) > 32:
            raise ValueError("counter bound")
        text = value.decode().strip()
        if text == "max":
            counters[key] = "unbounded"
        elif text.isascii() and text.isdecimal():
            counters[key] = int(text)
        else:
            raise ValueError("counter type")
    result["cgroup"] = {"status": "observed_metadata_only", **counters}
except (OSError, UnicodeError, ValueError):
    result["cgroup"] = {"status": "unavailable"}

print(json.dumps(result, indent=2, sort_keys=True))
